# Particle filter: notebook

The running example of the Note: a robot in a corridor that loops round a block, 10 m around, with three doors set back into the right-hand wall. A side range sensor reads 1.5 m in front of a door and 1.0 m beside plain wall (noise sd 0.15 m). We check every number in the Note, then run the experiments behind its figures.

## 0. The model

Map, sensor model, motion model, the two resamplers, the particle filter and an exact grid belief for comparison.

In [1]:
import numpy as np
import pandas as pd

import numpy as np

LOOP = 10.0                                         # corridor length (m); position wraps round at 10 m
DOORS = [(1.5, 2.5), (4.0, 5.0), (6.5, 7.5)]        # door recesses (m)
WALL, DOOR = 1.0, 1.5                               # side distance to the wall, and into a door recess (m)
SIGMA_Z = 0.15                                      # sensor noise, standard deviation (m)
SIGMA_U = 0.15                                      # motion noise per move, standard deviation (m)
U = 2.5                                             # commanded move (m)

X_TRUE = [2.1, 4.62, 7.08]                          # true positions at the three readings (moves 2.52, 2.46 m)
Z = [1.46, 1.58, 1.41]                              # the three side readings (m)


def h(x):
    """Map: the side distance the sensor should read at position x (m)."""
    x = np.mod(x, LOOP)
    d = np.full(np.shape(x), WALL, dtype=float)
    for a, b in DOORS:
        d = np.where((x >= a) & (x <= b), DOOR, d)
    return d


def likelihood(z, x, sigma=SIGMA_Z):
    """Sensor model p(z | x): height of the normal curve centred on the expected reading h(x), at the reading z."""
    return np.exp(-0.5 * ((z - h(x)) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def move(x, u, rng, sigma=SIGMA_U):
    """Motion model, sampled: each particle moves u plus its own random slip, and wraps round the loop."""
    return np.mod(x + u + rng.normal(0, sigma, np.shape(x)), LOOP)


def low_variance(w, rng, r=None):
    """Low-variance (systematic) resampling: one random start r in [0, 1/M), then M pointers 1/M apart.
    Returns the index of the particle each pointer lands on."""
    M = len(w)
    c = np.cumsum(w / np.sum(w))
    if r is None:
        r = rng.uniform(0, 1 / M)
    pointers = r + np.arange(M) / M
    return np.searchsorted(c, pointers, side="right").clip(max=M - 1)


def multinomial(w, rng):
    """Plain resampling: M independent random draws, each picking particle i with probability w_i."""
    c = np.cumsum(w / np.sum(w))
    return np.searchsorted(c, rng.uniform(0, 1, len(w)), side="right").clip(max=len(w) - 1)


def run(M, rng, readings=Z, moves=(U, U)):
    """One run of the particle filter on the three readings. Returns the stages for plotting:
    a list of (name, particles, weights) after each step."""
    x = rng.uniform(0, LOOP, M)
    w = np.full(M, 1 / M)
    stages = [("start", x.copy(), w.copy())]
    for k, z in enumerate(readings):
        if k > 0:
            x = move(x, moves[k - 1], rng)
            w = np.full(M, 1 / M)
            stages.append((f"move {k}", x.copy(), w.copy()))
        w = likelihood(z, x)
        w = w / w.sum()
        stages.append((f"weigh {k + 1}", x.copy(), w.copy()))
        idx = low_variance(w, rng)
        x, w = x[idx], np.full(M, 1 / M)
        stages.append((f"resample {k + 1}", x.copy(), w.copy()))
    return stages


def grid_belief(readings=Z, moves=(U, U), n=2000):
    """Exact belief on a fine grid (a histogram filter with n cells), for comparison with the particles.
    Returns the cell centres, the predicted belief before each reading (None before the first) and the belief
    after each reading, both as densities (per m)."""
    xs = (np.arange(n) + 0.5) * LOOP / n
    dx = LOOP / n
    bel = np.full(n, 1 / LOOP)
    out, preds = [], [None]
    # motion kernel: normal with mean u, sd SIGMA_U, wrapped round the loop
    for k, z in enumerate(readings):
        if k > 0:
            shift = xs[:, None] - xs[None, :] - moves[k - 1]           # destination minus source minus u
            shift = (shift + LOOP / 2) % LOOP - LOOP / 2
            kern = np.exp(-0.5 * (shift / SIGMA_U) ** 2) / (SIGMA_U * np.sqrt(2 * np.pi))
            bel = kern @ bel * dx
            preds.append(bel.copy())
        bel = likelihood(z, xs) * bel
        bel = bel / (bel.sum() * dx)
        out.append(bel.copy())
    return xs, preds, out


def kidnap_run(inject, seed, M=200, T=60, kid=30, step=0.5):
    """A long run: the robot drives 0.5 m per step (slip sd 0.03 m) and reads at every step. At step `kid` it is
    picked up and put down 5 m further on (the kidnapped robot). After each resampling, a share `inject` of the
    particles is replaced by random ones spread evenly. Returns the error of the estimate at every step (m)."""
    rng = np.random.default_rng(seed)
    xt, x, errs = 2.1, rng.uniform(0, LOOP, M), []
    for t in range(T):
        if t > 0:
            xt = (xt + step + rng.normal(0, 0.03)) % LOOP
            if t == kid:
                xt = (xt + 5) % LOOP
            x = move(x, step, rng, 0.03)
        z = h(np.array([xt]))[0] + rng.normal(0, SIGMA_Z)
        w = likelihood(z, x)
        x = x[low_variance(w / w.sum(), rng)]
        k = int(round(inject * M))
        if k:
            x[rng.choice(M, k, replace=False)] = rng.uniform(0, LOOP, k)
        ang = np.angle(np.mean(np.exp(2j * np.pi * x / LOOP))) % (2 * np.pi)   # mean round the loop
        est = ang * LOOP / (2 * np.pi)
        errs.append(abs((est - xt + LOOP / 2) % LOOP - LOOP / 2))
    return np.array(errs)

## 1. Weighing five particles by hand (Note, Section 4.2)

In [2]:
P = np.array([0.7, 1.6, 2.4, 3.3, 6.9])
lik = likelihood(1.46, P)          # height of the sensor curve at the reading
w = lik / lik.sum()                # normalised weights
pd.DataFrame(dict(position=P, expected_reading=h(P), density=lik.round(3), weight=w.round(4)))

,position,expected_reading,density,weight
0,0.7,1.0,0.024,0.0031
1,1.6,1.5,2.567,0.3313
2,2.4,1.5,2.567,0.3313
3,3.3,1.0,0.024,0.0031
4,6.9,1.5,2.567,0.3313


## 2. Resampling the five particles (Note, Section 4.3)

The low-variance sampler with the start r = 0.12, then plain resampling with five random numbers.

In [3]:
print("low-variance, r = 0.12:", np.bincount(low_variance(w, None, r=0.12), minlength=5))
darts = np.array([0.33, 0.99, 0.32, 0.79, 0.87])
print("plain, darts", darts, ":", np.bincount(np.searchsorted(np.cumsum(w), darts, side="right"), minlength=5))
print("chance plain resampling gives particle 3 no copy:", round((1 - w[2]) ** 5, 3))

low-variance, r = 0.12: [0 2 1 0 2]
plain, darts [0.33 0.99 0.32 0.79 0.87] : [0 2 0 0 3]
chance plain resampling gives particle 3 no copy: 0.134


How much does the number of copies of particle 3 vary from one resampling to the next? 100,000 repeats of each method.

In [4]:
rng = np.random.default_rng(0)
lv = [np.sum(low_variance(w, rng) == 2) for _ in range(100_000)]
mn = [np.sum(multinomial(w, rng) == 2) for _ in range(100_000)]
for name, c in (("low-variance", lv), ("plain", mn)):
    c = np.array(c)
    print(f"{name:13s} mean copies {c.mean():.3f}   sd {c.std():.3f}   share with no copy {np.mean(c == 0):.3f}")

low-variance  mean copies 1.655   sd 0.475   share with no copy 0.000
plain         mean copies 1.654   sd 1.052   share with no copy 0.135


## 3. One run of the filter, against the exact belief (Note, Section 4.5)

100 particles, seed 2 (the run of the animation). The share of weight at each door after each reading, from the particles and from the exact grid belief.

In [5]:
stages = run(100, np.random.default_rng(2))
xs, preds, bels = grid_belief()
dx = xs[1] - xs[0]
rows = []
for k in range(3):
    name, x, ww = stages[[s[0] for s in stages].index(f"weigh {k + 1}")]
    row = dict(reading=k + 1, z=Z[k], weighted_mean=round(np.sum(ww * x), 2))
    for d, (a, b) in enumerate(DOORS):
        row[f"door {d + 1} particles"] = round(np.sum(ww[(x >= a) & (x <= b)]), 2)
        row[f"door {d + 1} exact"] = round(bels[k][(xs >= a) & (xs <= b)].sum() * dx, 2)
    rows.append(row)
pd.DataFrame(rows)

,reading,z,weighted_mean,door 1 particles,door 1 exact,door 2 particles,door 2 exact,door 3 particles,door 3 exact
0,1,1.46,4.36,0.33,0.33,0.36,0.33,0.29,0.33
1,2,1.58,5.73,0.00,0.01,0.49,0.50,0.51,0.50
2,3,1.41,7.05,0.00,0.00,0.00,0.01,0.96,0.96


In [6]:
for k, b in enumerate(bels):
    m = np.sum(xs * b) * dx
    print(f"exact belief after reading {k + 1}: mean {m:.2f} m, sd {np.sqrt(np.sum((xs - m) ** 2 * b) * dx):.2f} m")

exact belief after reading 1: mean 4.52 m, sd 2.09 m
exact belief after reading 2: mean 5.73 m, sd 1.31 m
exact belief after reading 3: mean 7.03 m, sd 0.62 m


## 4. Monte Carlo accuracy and particle deprivation (Note, Sections 3.2 and 5.1)

1000 runs per particle count: the share of weight at door 3 after the third reading (exact: 0.956), and how often the filter ends at the right door.

In [7]:
exact3 = bels[2][(xs >= 6.5) & (xs <= 7.5)].sum() * dx
print("exact share at door 3:", round(exact3, 3))
rng = np.random.default_rng(1)
rows = []
for M in [5, 10, 20, 50, 100, 200, 1000]:
    share, ok = [], []
    for _ in range(1000):
        st = run(M, rng)
        _, x, ww = st[-2]
        share.append(np.sum(ww[(x >= 6.5) & (x <= 7.5)]))
        ok.append(np.mean(np.abs(st[-1][1] - 7.0) < 0.8) > 0.5)
    rows.append(dict(M=M, mean_share=round(np.mean(share), 3), sd_share=round(np.std(share), 3),
                     percent_right_door=round(100 * np.mean(ok), 1)))
pd.DataFrame(rows)

exact share at door 3: 0.956


,M,mean_share,sd_share,percent_right_door
0,5,0.432,0.485,43.9
1,10,0.612,0.466,63.0
2,20,0.835,0.315,87.8
3,50,0.942,0.076,99.6
4,100,0.950,0.030,100.0
5,200,0.953,0.020,100.0
6,1000,0.956,0.008,100.0


## 5. The kidnapped robot (Note, Section 5.2)

200 runs of 60 steps, 200 particles; at step 30 the robot is moved 5 m.

In [8]:
for inject in (0.0, 0.05):
    E = np.array([kidnap_run(inject, s) for s in range(200)])
    print(f"random share {inject:.2f}: mean error steps 25-29 {E[:, 25:30].mean():.2f} m,"
          f" runs recovered by step 59 {100 * np.mean(E[:, 59] < 0.5):.1f} percent")

random share 0.00: mean error steps 25-29 0.19 m, runs recovered by step 59 1.0 percent


random share 0.05: mean error steps 25-29 0.22 m, runs recovered by step 59 88.5 percent


## 6. Try it

Change `SIGMA_Z`, `SIGMA_U` or the number of particles and rerun sections 3 and 4.